In [1]:
!pip install pymupdf faiss-cpu pillow sentence-transformers transformers gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 61.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 64.1 MB/s eta 0:00:00


In [2]:
import fitz
import os
import numpy as np
from PIL import Image
from sentence_transformers import SentenceTransformer
import faiss
from transformers import T5ForConditionalGeneration, T5Tokenizer

In [4]:
file_path = "/content/sample_data/ARCS Data sample.pdf"

doc = fitz.open(file_path)

texts = []
image_paths = []

img_folder = "/content/pdf_images"
os.makedirs(img_folder, exist_ok=True)

for page_num in range(len(doc)):
    page = doc[page_num]
    texts.append(page.get_text())

    for i, img in enumerate(page.get_images(full=True)):
        xref = img[0]
        base = doc.extract_image(xref)
        img_bytes = base["image"]
        img_ext = base.get("ext", "png")

        path = f"{img_folder}/p{page_num}_{i}.{img_ext}"
        with open(path, "wb") as f:
            f.write(img_bytes)

        image_paths.append((page_num, path))

print(f"✅ Pages: {len(texts)}")
print(f"✅ Images: {len(image_paths)}")
print(f"✅ Sample text from page 0: {texts[0][:200]}")

✅ Pages: 407
✅ Images: 255
✅ Sample text from page 0: Oracle® Fusion Cloud EPM
Administering Account Reconciliation
E94355-83



In [5]:
def split_text(text, chunk_size=500, chunk_overlap=100):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start = end - chunk_overlap
    return chunks

chunks = []
chunk_page_map = []

for i, text in enumerate(texts):
    split_chunks = split_text(text)
    for c in split_chunks:
        chunks.append(c)
        chunk_page_map.append(i)

print(f"✅ Total chunks: {len(chunks)}")
print(f"✅ Sample chunk: {chunks[0][:200]}")

✅ Total chunks: 2080
✅ Sample chunk: Oracle® Fusion Cloud EPM
Administering Account Reconciliation
E94355-83



In [6]:
text_model = SentenceTransformer("all-mpnet-base-v2")
text_embeddings = text_model.encode(chunks, normalize_embeddings=True)

dim = text_embeddings.shape[1]
text_index = faiss.IndexFlatIP(dim)
text_index.add(np.array(text_embeddings))

print(f"✅ Text index: {text_index.ntotal} vectors")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Text index: 2080 vectors


In [7]:
clip_model = SentenceTransformer("clip-ViT-B-32")

image_embeddings = []
image_files = []

for page, path in image_paths:
    try:
        img = Image.open(path).convert("RGB")
        emb = clip_model.encode(img, normalize_embeddings=True)
        image_embeddings.append(emb)
        image_files.append((page, path))
    except:
        continue

image_embeddings = np.array(image_embeddings)

image_index = faiss.IndexFlatIP(image_embeddings.shape[1])
image_index.add(image_embeddings)

modules.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: /root/.cache/huggingface/hub/models--sentence-transformers--clip-ViT-B-32/snapshots/327ab6726d33c0e22f920c83f2ff9e4bd38ca37f/0_CLIPModel
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


In [8]:
from transformers import pipeline

In [9]:
qa_model = pipeline(
    "text-generation",
    model="google/flan-t5-base",
    max_new_tokens=200
)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLlamaF

In [10]:
chat_history = []

In [11]:
def chatbot(query):
    global chat_history

    # 🔹 TEXT SEARCH
    q_emb = text_model.encode([query], normalize_embeddings=True)
    D, I = text_index.search(np.array(q_emb), k=3)

    retrieved_chunks = [chunks[i] for i in I[0]]
    retrieved_pages = [chunk_page_map[i] for i in I[0]]

    context = "\n".join(retrieved_chunks)

    # 🔹 PROMPT
    prompt = f"""
You are an intelligent assistant.

Rules:
- Use ONLY the context
- If not found say: Not available in document
- Answer clearly

Context:
{context}

Question:
{query}
"""

    result = qa_model(prompt)[0]['generated_text']

    # 🔹 IMAGE FILTER (IMPORTANT FIX)
    q_img = clip_model.encode([query], normalize_embeddings=True)
    D_img, I_img = image_index.search(np.array(q_img), k=5)

    relevant_images = []

    for score, idx in zip(D_img[0], I_img[0]):
        if score > 0.25:  # 🔥 threshold fix
            relevant_images.append(image_files[idx])

    # 🔹 FILTER BY PAGE (IMPORTANT)
    final_images = [
        path for page, path in relevant_images if page in retrieved_pages
    ]

    # 🔹 SAVE CHAT HISTORY
    chat_history.append((query, result))
    chat_history = chat_history[-3:]

    return result, final_images, retrieved_pages

In [12]:
import gradio as gr

def chat_ui(query):
    answer, images, pages = chatbot(query)

    return answer, images, str(pages)

interface = gr.Interface(
    fn=chat_ui,
    inputs="text",
    outputs=["text", "gallery", "text"],
    title="📄 Multimodal PDF Chatbot"
)

interface.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://94ed878ec4c056be8e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
